In [1]:
import truststore
truststore.inject_into_ssl()

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

USER_AGENT environment variable not set, consider setting it to identify your requests.


Fixed size chunking

In [ ]:
from langchain_core.documents import Document

def chunk_fixed(docs, chunk_size = 512, overlap = 50) -> list[dict]:
    chunks = []
    step = chunk_size = overlap

    for doc in docs:
        words = doc.page_content.split()

        for i in range(0, len(words), step):
            chunk_words = " ".join(words[i: i+chunk_size])
            chunks.append(Document(
                page_content=chunk_words,
                metadata=doc.metadata
            ))

    return chunks

chunks = chunk_fixed(docs)

Sentence / Paragraph Chunking

In [6]:
import re
from langchain_core.documents import Document

def chunk_sentences(
    docs,
    sentences_per_chunk: int = 5,
    overlap_sentences: int = 1
) -> list[dict]:
    
    # sentence boundary pe split — lookbehind keeps the punctuation
    def split_sentences(text):
        return re.split(r'(?<=[.!?])\s+', text.strip())

    chunks = []

    for doc in docs:
        sentences = split_sentences(doc.page_content)
        step = sentences_per_chunk - overlap_sentences

        for i in range(0, len(sentences), step):
            batch = sentences[i : i + sentences_per_chunk]
            chunks.append({
                "text": " ".join(batch),
                "metadata": {
                    **doc.metadata,
                    "start_sentence": i,
                    "end_sentence": i + len(batch)
                }
            })
            if i + sentences_per_chunk >= len(sentences):
                break

    return chunks

chunks = chunk_sentences(docs)

Recursive Character Splitting

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,     
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]  # priority order
)

def chunk_recursive(docs):
    chunks = splitter.split_documents(docs)
    return chunks

chunks = chunk_recursive(docs)

Semantic Chunking

In [ ]:
# code from scratch

import numpy as np
from sentence_transformers import SentenceTransformer
import re

model = SentenceTransformer("all-MiniLM-L6-v2")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def semantic_chunk(docs, threshold: float = 0.75) -> list[dict]:
    chunks = []

    for doc in docs:
        # step 1: sentence split
        sentences = re.split(r'(?<=[.!?])\s+', doc.page_content.strip())
        if len(sentences) < 2:
            chunks.append({"text": doc.page_content, "metadata": doc.metadata})
            continue

        # step 2: embed all sentences
        embeddings = model.encode(sentences)

        # step 3: compute similarity between consecutive sentences
        similarities = [
            cosine_similarity(embeddings[i], embeddings[i+1])
            for i in range(len(embeddings) - 1)
        ]

        # step 4: split where similarity drops below threshold
        current_chunk = [sentences[0]]

        for i, sim in enumerate(similarities):
            if sim < threshold:
                # topic shift detected → save current, start new
                chunks.append({
                    "text": " ".join(current_chunk),
                    "metadata": {
                        **doc.metadata,
                        "similarity_at_split": round(sim, 3)
                    }
                })
                current_chunk = [sentences[i+1]]
            else:
                current_chunk.append(sentences[i+1])

        # last chunk
        if current_chunk:
            chunks.append({
                "text": " ".join(current_chunk),
                "metadata": doc.metadata
            })

    return chunks

chunks = semantic_chunk(docs)

/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8313.86it/s]


In [10]:
# build-in code
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

splitter = SemanticChunker(
    embeddings=HuggingFaceEmbeddings(),
    breakpoint_threshold_type="percentile", 
    breakpoint_threshold_amount=80
)

chunks = splitter.split_documents(docs)

/var/folders/_d/tj_f_hcs5gd3hx64vjm05drw0000gp/T/ipykernel_52384/1299336295.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7613.49it/s]


Parent-Child Chunking

In [ ]:
# code from scratch

import uuid
from langchain_text_splitters import RecursiveCharacterTextSplitter

def build_parent_child_chunks(docs) -> tuple[list[dict], list[dict]]:
    parent_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1024,
        chunk_overlap=100
    )
    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=256,
        chunk_overlap=30
    )

    parent_chunks = []
    child_chunks = []

    for doc in docs:
        parents = parent_splitter.split_documents([doc])

        for parent in parents:
            parent_id = str(uuid.uuid4()) 

            parent_chunks.append({
                "id": parent_id,
                "text": parent.page_content,
                "metadata": parent.metadata
            })

            children = child_splitter.split_documents([parent])

            for child in children:
                child_chunks.append({
                    "text": child.page_content,
                    "metadata": {
                        **child.metadata,
                        "parent_id": parent_id  
                    }
                })

    return parent_chunks, child_chunks


In [ ]:
parent_chunks = build_parent_child_chunks(docs)
parent_store = {}

for p in parent_chunks:
    parent_store[p["id"]] = p["text"]

def retrieve_with_parent(query: str, vectorstore, parent_store, k: int = 3) -> list[str]:
    results = vectorstore.similarity_search(query, k=k)

    seen = set()
    parent_texts = []

    for result in results:
        parent_id = result.metadata.get("parent_id")

        if parent_id and parent_id not in seen:
            seen.add(parent_id)
            parent_text = parent_store.get(parent_id)
            if parent_text:
                parent_texts.append(parent_text)

    return parent_texts


In [15]:
# Langchain Default Code
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

docstore = InMemoryStore()
vectorstore = Chroma(embedding_function=HuggingFaceEmbeddings())

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1024)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=256)

retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

retriever.add_documents(docs)
results = retriever.invoke("attention mechanism kya hai?")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8721.24it/s]
python(55325) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
